In [6]:
# === Core Utilities ===
import os
import logging
from collections import defaultdict
from typing import Tuple, List, Dict, Optional

# === Date/Time Handling ===
from datetime import datetime
# import datetime as dt  # Optional: alias if dt.datetime(...) is preferred

# === Numerical and Data Tools ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xc
import xskillscore as xs

import warnings

from ModelDataUtil import ModelDataReader

from ObservationDataUtil import ObsDataReader

from land_atmosphere_feedback import (
    LandAtmosphereFeedbackAnalyzer, 
    plot_lag_with_ci, 
    plot_feedback_map_with_sig
)

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures/obs_diag"

    # === Configuration and inputs ===
    exp_info = get_dart_config()
    exp_dict = exp_info['experiments'] 
    exp_cfg  = exp_info['global']
    
    reg_dict = {
        'NH':      'Northern Hemisphere', 
        'SH':      'Southern Hemisphere', 
        'Tropics': 'Tropics', 
        'NA':      'North America',
        'Global':  'global'
    }
    

    exp_list = ["CTRL10", "ERA510", "DART20", "DART40"]
    exp_path_base = "/compyfs/your_path/experiments"
    land_mask = xr.open_dataset(f"{out_path}/landmask_1x1.nc")["landfrac"] > 0.5

    feedback_analyzer = LandAtmosphereFeedbackAnalyzer(land_mask=land_mask)

    for exp in exp_list:
        print(f"Processing {exp}")
        exp_dir = os.path.join(exp_path_base, exp)
        members = load_member_data(exp_dir, var_list=["SMOIS", "PRECT"], use_dask=True)

        model_dict = {exp: members}
        results = feedback_analyzer.compute_lag_correlation_curve(
            model_dict, var_land="SMOIS", var_atm="PRECT", lags=list(range(-48, 49, 6))
        )
        
        feedback_analyzer.save_to_netcdf(results, f"lagcorr_{exp}_SMOIS_PRECT.nc")
    
    # Initialize
    analyzer = LandAtmosphereFeedbackAnalyzer(
        land_mask=land_frac > 0.5
    )

    # Example 1: Lag correlation curve (SMOIS → PRECT)
    curve = analyzer.compute_lag_correlation_curve(
        data_dict, "SMOIS", "PRECT", 
        lags=list(range(-48, 49, 6))
    )
    
    plot_lag_with_ci(curve, title="SMOIS → PRECT Feedback")

    # Example 2: Spatial map (TS → LHFLX at +24h)
    rmap = analyzer.compute_spatial_map(data_dict["DART20"], "TS", "LHFLX", lag=24)
    plot_feedback_map_with_sig(rmap, title="TS → LHFLX Spatial Feedback")

    # Example 3: Save curve to NetCDF
    analyzer.save_to_netcdf(curve, "land_feedback_SMOIS_PRECT.nc")